In [1]:
import sys

print(sys.executable)

e:\Projekt_GitHub\PortfolioProjekt\.venv\Scripts\python.exe


Damit ist bestätigt, dass Notebook die virtuelle Python-Umgebung des Projekts verwendet.

In [2]:
from pathlib import Path

import pandas as pd

Geprüft, ob pandas verfügbar ist.

In [3]:
DATA_DIR = Path("../data/cleaned")

DATA_DIR.exists()

True

Pfad in der Variablen DATA_DIR gespeichert und deren Existenz geprüft.

In [4]:
list(DATA_DIR.glob("*.csv"))

[WindowsPath('../data/cleaned/abschluesse_schulart.csv'),
 WindowsPath('../data/cleaned/oberschulen_oeffentlich_frei.csv'),
 WindowsPath('../data/cleaned/schuelerausgabensaetze_oberschule.csv'),
 WindowsPath('../data/cleaned/schulen_schueler_lehrer.csv')]

glob("*.csv") sucht alle Dateien im Ordner data/cleaned, deren Name auf .csv endet. Es werden dabei noch keine Daten eingelesen oder verändert.

# Die 1. Datei: oberschulen_oeffentlich_frei.csv einlesen und untersuchen.
Ergebnis:
68 Beobachtungen (Zeilen)
34 Schuljahre mit jeweils zwei Trägerschaften (öffentlich, frei)
keine mehrfach vorkommenden Kombinationen aus Schuljahr und Trägerschaft
ein fehlender Wert bei lehrpersonen_maennlich geprüft und durch 0 ersetzt
plausible Datentypen nach Bereinigung (str und int64)
die Daten in Spalte "traegerschaft" von str nach category geändert
Schülerzahlen intern konsistent
Lehrpersonenzahlen intern konsistent
keine fehlenden Werte nach Bereinigung

In [5]:
df_schulen = pd.read_csv(
    DATA_DIR / "oberschulen_oeffentlich_frei.csv",
    sep=";",
)
print(df_schulen.head().T)
print(df_schulen.shape)

                                 0           1           2           3  \
schuljahr                1992/1993   1993/1994   1994/1995   1995/1996   
traegerschaft           öffentlich  öffentlich  öffentlich  öffentlich   
schulen                        661         660         661         657   
schueler_gesamt             222966      216454      217118      220138   
schueler_maennlich          124441      120889      120315      120186   
schueler_weiblich            98525       95565       96803       99952   
lehrpersonen_gesamt          15338       14954       14985       14622   
lehrpersonen_maennlich      4930.0      4687.0      4717.0      4577.0   
lehrpersonen_weiblich        10408       10267       10268       10045   

                                 4  
schuljahr                1996/1997  
traegerschaft           öffentlich  
schulen                        653  
schueler_gesamt             222004  
schueler_maennlich          119757  
schueler_weiblich           102247  


sep=";" bedeutet: Die einzelnen Spalten der CSV sind durch Semikolons getrennt.
.T macht aus den Spalten Zeilen und umgekehrt
Wichtig: Das sind also keine einzelnen Schulen, sondern aggregierte Werte für eine Trägerform in einem Schuljahr.
Anzahl Zeilen(68),Spalten(9)

In [6]:
print(df_schulen["schuljahr"].unique())
print(df_schulen["traegerschaft"].unique())
df_schulen[["schuljahr", "traegerschaft"]].head(5)

<StringArray>
['1992/1993', '1993/1994', '1994/1995', '1995/1996', '1996/1997', '1997/1998',
 '1998/1999', '1999/2000', '2000/2001', '2001/2002', '2002/2003', '2003/2004',
 '2004/2005', '2005/2006', '2006/2007', '2007/2008', '2008/2009', '2009/2010',
 '2010/2011', '2011/2012', '2012/2013', '2013/2014', '2014/2015', '2015/2016',
 '2016/2017', '2017/2018', '2018/2019', '2019/2020', '2020/2021', '2021/2022',
 '2022/2023', '2023/2024', '2024/2025', '2025/2026']
Length: 34, dtype: str
<StringArray>
['öffentlich', 'frei']
Length: 2, dtype: str


,schuljahr,traegerschaft
0,1992/1993,öffentlich
1,1993/1994,öffentlich
2,1994/1995,öffentlich
3,1995/1996,öffentlich
4,1996/1997,öffentlich


es sind 34 + 34 = 68 Zeilen
Es werden aufgrund der bisherigen Struktur jeweils 34 Beobachtungen für öffentlich und frei erwartet.

In [7]:
df_schulen["traegerschaft"].value_counts()

traegerschaft
öffentlich    34
frei          34
Name: count, dtype: int64

In [8]:
df_schulen.dtypes

schuljahr                     str
traegerschaft                 str
schulen                     int64
schueler_gesamt             int64
schueler_maennlich          int64
schueler_weiblich           int64
lehrpersonen_gesamt         int64
lehrpersonen_maennlich    float64
lehrpersonen_weiblich       int64
dtype: object

In [9]:
print(df_schulen.isna().sum())
df_schulen[df_schulen["lehrpersonen_maennlich"].isna()]

schuljahr                 0
traegerschaft             0
schulen                   0
schueler_gesamt           0
schueler_maennlich        0
schueler_weiblich         0
lehrpersonen_gesamt       0
lehrpersonen_maennlich    1
lehrpersonen_weiblich     0
dtype: int64


,schuljahr,traegerschaft,schulen,schueler_gesamt,schueler_maennlich,schueler_weiblich,lehrpersonen_gesamt,lehrpersonen_maennlich,lehrpersonen_weiblich
34,1992/1993,frei,1,81,57,24,4,NaN,4


Datentyp float ist verdächtig, deshalb auf fehlende Werte hin prüfen
Es gibt genau einmal NaN in Zeile 34.

In [10]:
df_schulen.loc[34]

schuljahr                 1992/1993
traegerschaft                  frei
schulen                           1
schueler_gesamt                  81
schueler_maennlich               57
schueler_weiblich                24
lehrpersonen_gesamt               4
lehrpersonen_maennlich          NaN
lehrpersonen_weiblich             4
Name: 34, dtype: object

Es gibt nur 4x weiblich aber kein männlich --> NaN durch "0" ersetzen

In [11]:
df_schulen.loc[34, "lehrpersonen_maennlich"] = 0
df_schulen["lehrpersonen_maennlich"] = (
    df_schulen["lehrpersonen_maennlich"].astype("int64")
)
df_schulen["traegerschaft"] = (
    df_schulen["traegerschaft"].astype("category")
)
print(df_schulen.dtypes)

schuljahr                      str
traegerschaft             category
schulen                      int64
schueler_gesamt              int64
schueler_maennlich           int64
schueler_weiblich            int64
lehrpersonen_gesamt          int64
lehrpersonen_maennlich       int64
lehrpersonen_weiblich        int64
dtype: object


NaN durch "0" ersetzt und Datentyp geändert

In [12]:
print(df_schulen.duplicated().sum())
df_schulen.duplicated(
    subset=["schuljahr", "traegerschaft"]
).sum()

0


np.int64(0)

Es gibt keine doppelten Zeilen.
Werte der beiden Spalten kommen auch nicht mehrfach vor.

In [13]:
print((
    df_schulen["schueler_gesamt"]
    == df_schulen["schueler_maennlich"] + df_schulen["schueler_weiblich"]
).value_counts())
(
    df_schulen["lehrpersonen_gesamt"]
    == df_schulen["lehrpersonen_maennlich"]
    + df_schulen["lehrpersonen_weiblich"]
).value_counts()

True    68
Name: count, dtype: int64


True    68
Name: count, dtype: int64

Prüfung:
Schüler gesamt=mannlich+weiblich
Lehrpersonen gesamt=maännlich+weiblich

In [14]:
df_schulen.info()

<class 'pandas.DataFrame'>
RangeIndex: 68 entries, 0 to 67
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   schuljahr               68 non-null     str     
 1   traegerschaft           68 non-null     category
 2   schulen                 68 non-null     int64   
 3   schueler_gesamt         68 non-null     int64   
 4   schueler_maennlich      68 non-null     int64   
 5   schueler_weiblich       68 non-null     int64   
 6   lehrpersonen_gesamt     68 non-null     int64   
 7   lehrpersonen_maennlich  68 non-null     int64   
 8   lehrpersonen_weiblich   68 non-null     int64   
dtypes: category(1), int64(7), str(1)
memory usage: 4.5 KB


## Daten aus dem Dataframe df_schulen in der csv-Datei abspeichern

In [44]:
df_schulen.to_csv(
    "../data/cleaned/oberschulen_oeffentlich_frei.csv",
    sep=";", # Simikolon als Trennzeichen
    index=False, # Index nicht mit abspeichern
    encoding="utf-8" # UTF-8 als Zeichensatz verwenden
)

Kontrolle:

In [45]:
df_schulen_test = pd.read_csv(
    "../data/cleaned/oberschulen_oeffentlich_frei.csv",
    sep=";"
)

df_schulen_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 68 entries, 0 to 67
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   schuljahr               68 non-null     str  
 1   traegerschaft           68 non-null     str  
 2   schulen                 68 non-null     int64
 3   schueler_gesamt         68 non-null     int64
 4   schueler_maennlich      68 non-null     int64
 5   schueler_weiblich       68 non-null     int64
 6   lehrpersonen_gesamt     68 non-null     int64
 7   lehrpersonen_maennlich  68 non-null     int64
 8   lehrpersonen_weiblich   68 non-null     int64
dtypes: int64(7), str(2)
memory usage: 4.9 KB


Ergebnis: in csv werden keine pandas Datentypen gespeichert --> beim Einlesen "dtype={"traegerschaft": "category"}" verwenden

# Die 2. Datei: schulen_schueler_lehrer.csv einlesen und untersuchen.
Ergebnis:
31 Beobachtungen (Zeilen)
Spalte "schulart" geprüft und wegen konstantem Inhalt im DataFrame entfernt
keine fehlenden Werte (kein NaN)
plausible Datentypen (str und int64)
Schülerzahlen intern konsistent
Lehrpersonenzahlen intern konsistent
keine mehrfach vorkommenden Schuljahre

In [15]:
df_schueler_lehrer = pd.read_csv(
    DATA_DIR / "schulen_schueler_lehrer.csv"
)

Datei wurde eingelesen

In [16]:
display(df_schueler_lehrer.head().T)

print("Dimension:", df_schueler_lehrer.shape)

df_schueler_lehrer.info()

,0,1,2,3,4
schulart,Schularten mit mehreren Bildungsgängen,Schularten mit mehreren Bildungsgängen,Schularten mit mehreren Bildungsgängen,Schularten mit mehreren Bildungsgängen,Schularten mit mehreren Bildungsgängen
schuljahr,1995/96,1996/97,1997/98,1998/99,1999/00
schulen_anzahl,659,657,651,648,643
klassen_anzahl,9471,9322,9210,9170,9033
schueler_gesamt,220371,222608,221100,218147,214149
schueler_maennlich,120332,120119,118628,116560,113999
schueler_weiblich,100039,102489,102472,101587,100150
lehrpersonen_gesamt,14634,14171,14347,14271,14015
lehrpersonen_maennlich,4580,4344,4366,4327,4256
lehrpersonen_weiblich,10054,9827,9981,9944,9759


Dimension: (31, 10)
<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   schulart                31 non-null     str  
 1   schuljahr               31 non-null     str  
 2   schulen_anzahl          31 non-null     int64
 3   klassen_anzahl          31 non-null     int64
 4   schueler_gesamt         31 non-null     int64
 5   schueler_maennlich      31 non-null     int64
 6   schueler_weiblich       31 non-null     int64
 7   lehrpersonen_gesamt     31 non-null     int64
 8   lehrpersonen_maennlich  31 non-null     int64
 9   lehrpersonen_weiblich   31 non-null     int64
dtypes: int64(8), str(2)
memory usage: 2.6 KB


In [17]:
df_schueler_lehrer["schulart"].unique()

<StringArray>
['Schularten mit mehreren Bildungsgängen']
Length: 1, dtype: str

'Schularten mit mehreren Bildungsgängen' kommt nur einmal vor und kann im Dataframe gelöscht werden (Datei unverändert!) gelöscht werden.

In [18]:
df_schueler_lehrer = df_schueler_lehrer.drop(columns="schulart")

In [19]:
df_schueler_lehrer.info() # Änderungen prüfen

<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   schuljahr               31 non-null     str  
 1   schulen_anzahl          31 non-null     int64
 2   klassen_anzahl          31 non-null     int64
 3   schueler_gesamt         31 non-null     int64
 4   schueler_maennlich      31 non-null     int64
 5   schueler_weiblich       31 non-null     int64
 6   lehrpersonen_gesamt     31 non-null     int64
 7   lehrpersonen_maennlich  31 non-null     int64
 8   lehrpersonen_weiblich   31 non-null     int64
dtypes: int64(8), str(1)
memory usage: 2.3 KB


## Konsistenz der Daten überprüfen

In [20]:
print((
    df_schueler_lehrer["schueler_gesamt"]
    == df_schueler_lehrer["schueler_maennlich"]
    + df_schueler_lehrer["schueler_weiblich"]
).value_counts())
(
    df_schueler_lehrer["lehrpersonen_gesamt"]
    == df_schueler_lehrer["lehrpersonen_maennlich"]
    + df_schueler_lehrer["lehrpersonen_weiblich"]
).value_counts()


True    31
Name: count, dtype: int64


True    31
Name: count, dtype: int64

bestätigt: 
Schüler gesamt=männlich+weiblich und
Lehrpersonen gesamt=männlich+weiblich

## Prüfen, ob ein Schuljahr doppelt vorkommt

In [21]:
df_schueler_lehrer.duplicated(subset=["schuljahr"]).sum() # Ergenis: Nein

np.int64(0)

In [46]:
df_schueler_lehrer.info() # Datentypen prüfen

<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   schuljahr               31 non-null     str  
 1   schulen_anzahl          31 non-null     int64
 2   klassen_anzahl          31 non-null     int64
 3   schueler_gesamt         31 non-null     int64
 4   schueler_maennlich      31 non-null     int64
 5   schueler_weiblich       31 non-null     int64
 6   lehrpersonen_gesamt     31 non-null     int64
 7   lehrpersonen_maennlich  31 non-null     int64
 8   lehrpersonen_weiblich   31 non-null     int64
dtypes: int64(8), str(1)
memory usage: 2.3 KB


## Dataframe df_schueler_lehrer in csv Datei abspeichern und Erfolg prüfen:


In [ ]:
df_schueler_lehrer.to_csv(
    "../data/cleaned/schulen_schueler_lehrer.csv",
    sep=";",
    index=False,
    encoding="utf-8"
)

<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   schuljahr               31 non-null     str  
 1   schulen_anzahl          31 non-null     int64
 2   klassen_anzahl          31 non-null     int64
 3   schueler_gesamt         31 non-null     int64
 4   schueler_maennlich      31 non-null     int64
 5   schueler_weiblich       31 non-null     int64
 6   lehrpersonen_gesamt     31 non-null     int64
 7   lehrpersonen_maennlich  31 non-null     int64
 8   lehrpersonen_weiblich   31 non-null     int64
dtypes: int64(8), str(1)
memory usage: 2.3 KB


Prüfen:

In [48]:
df_schueler_lehrer_test = pd.read_csv(
    "../data/cleaned/schulen_schueler_lehrer.csv",
    sep=";"
)

df_schueler_lehrer_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   schuljahr               31 non-null     str  
 1   schulen_anzahl          31 non-null     int64
 2   klassen_anzahl          31 non-null     int64
 3   schueler_gesamt         31 non-null     int64
 4   schueler_maennlich      31 non-null     int64
 5   schueler_weiblich       31 non-null     int64
 6   lehrpersonen_gesamt     31 non-null     int64
 7   lehrpersonen_maennlich  31 non-null     int64
 8   lehrpersonen_weiblich   31 non-null     int64
dtypes: int64(8), str(1)
memory usage: 2.3 KB


# Die 3. Datei: "abschluesse_schulart.csv" einlesen und prüfen.

Ergebnis: 5.022 Beobachtungen im vollständigen Datensatz
558 Beobachtungen für Schularten mit mehreren Bildungsgängen
relevante Schulart für die Analyse als Mittel-/Oberschulen identifiziert und separat gefiltert
Originalbezeichnung der Schulart im Ausgangs-DataFrame beibehalten
GENESIS-Sonderwerte - und x geprüft
- für die Analyse als 0 behandelt
x als nicht numerisch anwendbar (<NA>) behandelt
zusätzliche Spalte anzahl_numerisch als Int64 angelegt
abschlussart und geschlecht als category definiert
keine mehrfach vorkommenden Kombinationen aus Schuljahr, Abschlussart, Geschlecht und Schulart
Zeitraum: 1994/95 bis 2024/25
62 <NA> im gefilterten Oberschul-Datensatz; diese entsprechen den zuvor identifizierten x-Konstellationen

In [22]:
df_abschluesse = pd.read_csv(
    DATA_DIR / "abschluesse_schulart.csv"
)
display(df_abschluesse.head().T)
print("Dimension:", df_abschluesse.shape)
df_abschluesse.info()

,0,1,2,3,4
schuljahr,1994/95,1994/95,1994/95,1994/95,1994/95
abschlussart,ohne Hauptschulabschluss,ohne Hauptschulabschluss,ohne Hauptschulabschluss,ohne Hauptschulabschluss,ohne Hauptschulabschluss
geschlecht,männlich,männlich,männlich,männlich,männlich
schulart,Grundschulen,Schularten mit mehreren Bildungsgängen,Gymnasien,Förderschulen,Freie Waldorfschulen
anzahl,-,2317,274,1683,-


Dimension: (5022, 5)
<class 'pandas.DataFrame'>
RangeIndex: 5022 entries, 0 to 5021
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   schuljahr     5022 non-null   str  
 1   abschlussart  5022 non-null   str  
 2   geschlecht    5022 non-null   str  
 3   schulart      5022 non-null   str  
 4   anzahl        5022 non-null   str  
dtypes: str(5)
memory usage: 196.3 KB


In [23]:
df_abschluesse["anzahl"].value_counts().head(20) # zählen wie oft jeder unterschiedliche Wert vorkommt

anzahl
-     2835
x      206
1       49
2       41
3       29
4       26
7       19
25      19
27      18
20      17
9       16
10      16
16      16
6       15
13      15
17      15
18      14
11      14
31      14
22      13
Name: count, dtype: int64

Ergebnis: insgesamt 2835+206 Sonderwerte --> prüfen für was die Zeichen "-" und "x" stehen.

In [24]:
df_abschluesse[df_abschluesse["anzahl"] == "-"].head(10) # mit df_abschluesse[...] betr. Zeilen filtern

,schuljahr,abschlussart,geschlecht,schulart,anzahl
0,1994/95,ohne Hauptschulabschluss,männlich,Grundschulen,-
4,1994/95,ohne Hauptschulabschluss,männlich,Freie Waldorfschulen,-
5,1994/95,ohne Hauptschulabschluss,männlich,Gemeinschaftsschulen,-
6,1994/95,ohne Hauptschulabschluss,männlich,Integrierte Gesamtschulen,-
8,1994/95,ohne Hauptschulabschluss,männlich,außerdem Schulkindergärten entspr.Vorbereitung...,-
9,1995/96,ohne Hauptschulabschluss,männlich,Grundschulen,-
14,1995/96,ohne Hauptschulabschluss,männlich,Gemeinschaftsschulen,-
15,1995/96,ohne Hauptschulabschluss,männlich,Integrierte Gesamtschulen,-
17,1995/96,ohne Hauptschulabschluss,männlich,außerdem Schulkindergärten entspr.Vorbereitung...,-
18,1996/97,ohne Hauptschulabschluss,männlich,Grundschulen,-


Dem Ergebnis nach vermutlich "-" nur für "ohne Hauptschulabschluss" --> prüfen

In [25]:
df_abschluesse.loc[
    df_abschluesse["anzahl"] == "-",
    "abschlussart"
].value_counts() # Spalte "anzahl" nur Abschlussart "-" betrachtet und gezählt

abschlussart
mit Fachhochschulreife            837
ohne Hauptschulabschluss          441
mit allgemeiner Hochschulreife    429
mit Hauptschulabschluss           393
mit Realschulabschluss            369
Insgesamt                         366
Name: count, dtype: int64

## Ergebnis der Prüfung: es betrifft auch andere Abschlüsse --> ürsprüngliche Quellenbeschreibung GENESIS prüfen
Es gibt eine offizielle Zeichenerklärung des Statistischen Landesamtes Sachsen, in den Metadaten der Ursprungsdatei war nichts konkretes zu finden:
- = „Genau Null oder ggf. zur Sicherstellung der statistischen Geheimhaltung auf Null geändert“
x = „Tabellenfach gesperrt, weil Aussage nicht sinnvoll“
Mögliche Maßnahme -->
"-" durch "0" ersetzen und
"x" gesondert prüfen um zu verstehen, warum bestimmte Zeilen gesperrt sind

In [26]:
df_abschluesse.loc[
    df_abschluesse["anzahl"] == "x"
].head(3)

,schuljahr,abschlussart,geschlecht,schulart,anzahl
7,1994/95,ohne Hauptschulabschluss,männlich,Schulen des zweiten Bildungsweges,x
16,1995/96,ohne Hauptschulabschluss,männlich,Schulen des zweiten Bildungsweges,x
25,1996/97,ohne Hauptschulabschluss,männlich,Schulen des zweiten Bildungsweges,x


Ergebnis: Die Inhalte scheinen pro Schuljahr identisch zu sein --> für alle 206 Zeilen prüfen

In [27]:
df_abschluesse.loc[
    df_abschluesse["anzahl"] == "x",
    ["abschlussart", "geschlecht", "schulart"]
].value_counts()

abschlussart                    geschlecht  schulart                              
ohne Hauptschulabschluss        männlich    Schulen des zweiten Bildungsweges         31
                                weiblich    Schulen des zweiten Bildungsweges         31
mit allgemeiner Hochschulreife  männlich    Schularten mit mehreren Bildungsgängen    31
                                            Förderschulen                             31
                                weiblich    Schularten mit mehreren Bildungsgängen    31
                                            Förderschulen                             31
mit Hauptschulabschluss         männlich    Gymnasien                                 10
                                weiblich    Gymnasien                                 10
Name: count, dtype: int64

Ergebnis: Es handelt sich um Kombinationen von Abschlussart und Schulart, für die eine Angabe offenbar nicht sinnvoll ist.--> damit Informationen nicht verloren gehen, wirde eine weitere numerische Spalte angelegt um damit weiter zu arbeiten

In [28]:
df_abschluesse["anzahl_numerisch"] = pd.to_numeric(
    df_abschluesse["anzahl"].replace("-", "0"),
    errors="coerce",
) # Zahl-->Zahl, "-"-->"0" und "x"-->NaN (coerce)
df_abschluesse["anzahl_numerisch"] = (
    df_abschluesse["anzahl_numerisch"].astype("Int64")
) # Int64 kann NaN-Werte enthalten, im Gegensatz zu int64, das nur ganze Zahlen enthält

In [29]:
df_abschluesse[["anzahl", "anzahl_numerisch"]].value_counts(
    dropna=False
).head(3)

anzahl  anzahl_numerisch
-       0                   2835
x       <NA>                 206
1       1                     49
Name: count, dtype: int64

Überprüfung, ob die Umwandlung für alle Werte der Spalte"anzahl" funktioniert hat:

In [30]:
print(df_abschluesse["anzahl_numerisch"].isna().sum()) # zählt NaN-Werte in der Spalte "anzahl_numerisch"
(df_abschluesse["anzahl_numerisch"] == 0).sum() # zählt wie oft der Wert 0 in der Spalte "anzahl_numerisch" vorkommt

206


np.int64(2835)

Wir prüfen deshalb, ob diese Kombination mehrfach vorkommt:

In [31]:
df_abschluesse.duplicated(
    subset=["schuljahr", "abschlussart", "geschlecht", "schulart"]
).sum()

np.int64(0)

Zeitraum und Kategorien prüfen:

In [32]:
print(df_abschluesse["schuljahr"].iloc[0])
print(df_abschluesse["schuljahr"].iloc[-1]) # Zeitreaum prüfen, indem man die erste und letzte Ze

print(df_abschluesse["abschlussart"].unique())
print(df_abschluesse["geschlecht"].unique()) 
df_abschluesse["schulart"].unique()# Kategorien prüfen

1994/95
2024/25
<StringArray>
[      'ohne Hauptschulabschluss',        'mit Hauptschulabschluss',
         'mit Realschulabschluss',         'mit Fachhochschulreife',
 'mit allgemeiner Hochschulreife',                      'Insgesamt']
Length: 6, dtype: str
<StringArray>
['männlich', 'weiblich', 'Insgesamt']
Length: 3, dtype: str


<StringArray>
[                                      'Grundschulen',
             'Schularten mit mehreren Bildungsgängen',
                                          'Gymnasien',
                                      'Förderschulen',
                               'Freie Waldorfschulen',
                               'Gemeinschaftsschulen',
                          'Integrierte Gesamtschulen',
                  'Schulen des zweiten Bildungsweges',
 'außerdem Schulkindergärten entspr.Vorbereitungskl.']
Length: 9, dtype: str

Ergtebnis: 31 Schuljahre; "Insgesamt" ist keine Abschlussart
Gesamtschulen sollen nicht mit Oberschulen gleichgesetzt werden, auch wenn an diesen Schulen die Abschlüsse einer Oberschule möglich sind

In [33]:
print(df_abschluesse.loc[
    df_abschluesse["abschlussart"] == "Insgesamt"
].head(3) )# 3 ersten Zeilen mit "Insgesamt" in der Spalte "abschlussart" anzeigen
(df_abschluesse["abschlussart"]=="Insgesamt").sum() # wie oft kommen Zeieln mit "Insgesamt" in der Spalte "abschlussart" vor

     schuljahr abschlussart geschlecht  \
4185   1994/95    Insgesamt   männlich   
4186   1994/95    Insgesamt   männlich   
4187   1994/95    Insgesamt   männlich   

                                    schulart anzahl  anzahl_numerisch  
4185                            Grundschulen      -                 0  
4186  Schularten mit mehreren Bildungsgängen  22156             22156  
4187                               Gymnasien   6269              6269  


np.int64(837)

## Zuordnung zur Oberschule

Die amtliche Kategorie `Schularten mit mehreren Bildungsgängen` wird für die weitere
Analyse als relevante Kategorie für sächsische Mittel-/Oberschulen verwendet.

Die Originalbezeichnung aus dem Orginaldatensatz bleibt. Für die weitere
Analyse wird daraus ein eigener DataFrame `df_abschluesse_oberschulen` erzeugt.

In [34]:
df_abschluesse_oberschulen = df_abschluesse[
    df_abschluesse["schulart"] == "Schularten mit mehreren Bildungsgängen"
].copy() # neuen DataFrame df_abschluesse_oberschulen erstellen, der nur die Zeilen aus df_abschluesse enthält, bei denen die Spalte "schulart" den Wert "Schularten mit mehreren Bildungsgängen" hat. Die Methode copy() wird verwendet, um eine Kopie der gefilterten Daten zu erstellen, sodass Änderungen an df_abschluesse_oberschulen nicht df_abschluesse beeinflussen.

".copy" um einen eigenständigen und unabhängigen Datansatz zu erstellen, losgelöst von df_abschlüsse (saubere Trennung)

Überprüfung des neuen Dataframe:

In [35]:
print("Anzahl der Zeilen und Spalten:", df_abschluesse_oberschulen.shape)
print("Werteanzahl der Spalte 'schulart':", df_abschluesse_oberschulen["schulart"].value_counts())
df_abschluesse_oberschulen.info()

Anzahl der Zeilen und Spalten: (558, 6)
Werteanzahl der Spalte 'schulart': schulart
Schularten mit mehreren Bildungsgängen    558
Name: count, dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 558 entries, 1 to 5014
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   schuljahr         558 non-null    str  
 1   abschlussart      558 non-null    str  
 2   geschlecht        558 non-null    str  
 3   schulart          558 non-null    str  
 4   anzahl            558 non-null    str  
 5   anzahl_numerisch  496 non-null    Int64
dtypes: Int64(1), str(5)
memory usage: 26.8 KB


Die Spalte Schulart kann entfernt werden, da nur noch "Oberschule" enthalten ist und der Datentyp für aaaaaAbschlussart und Geschlecht, kann/sollte in "Kategorie" umgeändert werden

In [36]:
df_abschluesse_oberschulen = (
    df_abschluesse_oberschulen.drop(columns="schulart") # "Spalte "schulart" wird entfernt, da alle Zeilen den gleichen Wert haben und diese Information redundant ist
)
df_abschluesse_oberschulen["abschlussart"] = (
    df_abschluesse_oberschulen["abschlussart"].astype("category")
)
df_abschluesse_oberschulen["geschlecht"] = (
    df_abschluesse_oberschulen["geschlecht"].astype("category")
)
df_abschluesse_oberschulen.info() # Änderungen prüfen

<class 'pandas.DataFrame'>
RangeIndex: 558 entries, 1 to 5014
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   schuljahr         558 non-null    str     
 1   abschlussart      558 non-null    category
 2   geschlecht        558 non-null    category
 3   anzahl            558 non-null    str     
 4   anzahl_numerisch  496 non-null    Int64   
dtypes: Int64(1), category(2), str(2)
memory usage: 14.9 KB


## Dataframe df_abschlüsse_oberschulen in csv speichern und prüfen

In [49]:
df_abschluesse_oberschulen.to_csv(
    "../data/cleaned/abschluesse_schulart.csv",
    sep=";",
    index=False,
    encoding="utf-8"
)

Prüfen:

In [50]:
df_abschluesse_oberschulen_test = pd.read_csv(
    "../data/cleaned/abschluesse_schulart.csv",
    sep=";"
)

df_abschluesse_oberschulen_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 558 entries, 0 to 557
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   schuljahr         558 non-null    str    
 1   abschlussart      558 non-null    str    
 2   geschlecht        558 non-null    str    
 3   anzahl            558 non-null    str    
 4   anzahl_numerisch  496 non-null    float64
dtypes: float64(1), str(4)
memory usage: 21.9 KB


"category" wie erwartet nicht in csv abgespeichert und jetzt auch float64 statt int64 --> beim Einlesen beachten

# Die 4. Datei: "schuelerausgabensaetze_oberschule.csv" einlesen und prüfen.
Ergebnis: 8 Beobachtungen
Zeitraum 2018/19 bis 2025/26
keine fehlenden Werte
plausible Datentypen
schulart geprüft und wegen konstantem Inhalt im DataFrame entfernt
status geprüft und wegen konstantem Inhalt im DataFrame entfernt
schuelerausgabensatz_eur als float64
keine mehrfach vorkommenden Schuljahre
eine Beobachtung je Schuljahr mit dem jeweiligen Schülerausgabensatz in Euro

In [37]:
df_finanzierung = pd.read_csv(
    DATA_DIR / "schuelerausgabensaetze_oberschule.csv",
    sep=";",
    decimal=",",
)
display(df_finanzierung.head().T)
print("Zeilenanzahl, Spaltenanzahl:", df_finanzierung.shape)
df_finanzierung.info()

,0,1,2,3,4
schuljahr,2018/19,2019/20,2020/21,2021/22,2022/23
schulart,Oberschule,Oberschule,Oberschule,Oberschule,Oberschule
schuelerausgabensatz_eur,6027.27,6431.76,6591.08,6698.22,7058.01
status,endgültig,endgültig,endgültig,endgültig,endgültig


Zeilenanzahl, Spaltenanzahl: (8, 4)
<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   schuljahr                 8 non-null      str    
 1   schulart                  8 non-null      str    
 2   schuelerausgabensatz_eur  8 non-null      float64
 3   status                    8 non-null      str    
dtypes: float64(1), str(3)
memory usage: 388.0 bytes


In [38]:
df_finanzierung = pd.read_csv(
    DATA_DIR / "schuelerausgabensaetze_oberschule.csv",
    sep=";",
    decimal=",",
)
display(df_finanzierung.head().T)

print("Dimension:", df_finanzierung.shape)

df_finanzierung.info()

,0,1,2,3,4
schuljahr,2018/19,2019/20,2020/21,2021/22,2022/23
schulart,Oberschule,Oberschule,Oberschule,Oberschule,Oberschule
schuelerausgabensatz_eur,6027.27,6431.76,6591.08,6698.22,7058.01
status,endgültig,endgültig,endgültig,endgültig,endgültig


Dimension: (8, 4)
<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   schuljahr                 8 non-null      str    
 1   schulart                  8 non-null      str    
 2   schuelerausgabensatz_eur  8 non-null      float64
 3   status                    8 non-null      str    
dtypes: float64(1), str(3)
memory usage: 388.0 bytes


Zu klärende Fragen: Ist "schulart" identisch in allen Zeilen und was steht in "status".

In [39]:
print(df_finanzierung["schulart"].value_counts())

print(df_finanzierung["status"].value_counts())

schulart
Oberschule    8
Name: count, dtype: int64
status
endgültig    8
Name: count, dtype: int64


Beide Spalten bieten konstant gleiche Inhalte und keine relevanten zusätzlichen Informationen --> können aus dem Datensatz entfernt werden.

In [40]:
df_finanzierung = df_finanzierung.drop(
    columns=["schulart", "status"]
) # Löschen der Spalten "schulart" und "status", da diese Informationen redundant sind, da alle Zeilen den gleichen Wert haben
df_finanzierung.info() # Prüfen, ob die Spalten erfolgreich gelöscht wurden

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 2 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   schuljahr                 8 non-null      str    
 1   schuelerausgabensatz_eur  8 non-null      float64
dtypes: float64(1), str(1)
memory usage: 260.0 bytes


In [41]:
df_finanzierung.duplicated(
    subset=["schuljahr"]
).sum() # Prüfen, ob es doppelte Zeilen gibt, die sich nur im Schuljahr unterscheiden. Ergebnis: Nein, keine doppelten Zeilen

np.int64(0)

In [ ]:
print(df_finanzierung["schuljahr"].iloc[0])
print(df_finanzierung["schuljahr"].iloc[-1]) # Zeitreaum prüfen, indem man die erste und letzte Zeile der Spalte "schuljahr" ausgibt

2018/19
2025/26


## Datatframe in csv speichern und prüfen

In [51]:
df_finanzierung.to_csv(
    "../data/cleaned/schuelerausgabensaetze_oberschule.csv",
    sep=";",
    index=False,
    encoding="utf-8"
)

Prüfen:

In [52]:
df_finanzierung_test = pd.read_csv(
    "../data/cleaned/schuelerausgabensaetze_oberschule.csv",
    sep=";"
)

df_finanzierung_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 2 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   schuljahr                 8 non-null      str    
 1   schuelerausgabensatz_eur  8 non-null      float64
dtypes: float64(1), str(1)
memory usage: 260.0 bytes
